# 다변량 Empirical-Bayes

gene×event-type의 class weight 행렬을 rank 4/8 저랭크 공유 구조로 수축합니다. P1+EB의 토큰 정의는 유지하고, 희귀 암종 간 정보 공유만 검증합니다.

seed 42 screen만 수행합니다. 사전 승격 조건(+0.010, 4/5 fold 양수, 안전성 통과)을 만족한 하나의 구성만 3-seed로 확장합니다.


In [ ]:
from pathlib import Path
import subprocess, sys
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "experiments/gs/notebooks/exp_model_003/common/run_p1_eb_axis.py").exists())
COMMON = ROOT / "experiments/gs/notebooks/exp_model_003/common"
RESULT = ROOT / "experiments/gs/notebooks/exp_model_003/result"
RUNNER = COMMON / "run_p1_eb_axis.py"
assert RUNNER.exists(), RUNNER


In [ ]:
SEED = 42
RUN_EXPERIMENT = True  # 전체 OOF 실행 전에는 False로 둘 수 있습니다.
RUN_ID = "exp-multivariate-eb-01"

if RUN_EXPERIMENT:
    command = [sys.executable, str(RUNNER), "--axis", "multieb", "--seed", str(SEED), "--run-id", RUN_ID]
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in tqdm(process.stdout, desc="OOF runner", unit="line"):
        print(line, end="")
        tail = (tail + [line])[-120:]
    return_code = process.wait()
    if return_code:
        raise RuntimeError(f"runner failed (exit={return_code}). Last runner output:\n" + "".join(tail))
else:
    print("RUN_EXPERIMENT=False: 기존 결과만 조회합니다.")


In [ ]:
summary = pd.read_csv(RESULT / f"exp-multivariate-eb-01_seed{SEED}_summary.csv")
folds = pd.read_csv(RESULT / f"exp-multivariate-eb-01_seed{SEED}_fold_metrics.csv")
display(summary); display(folds)
assert summary.leakage_check.all()
assert summary.nan_as_mutation_count.eq(0).all()
summary.set_index("variant")["oof_macro_f1"].plot.bar(figsize=(8, 4), title="다변량 Empirical-Bayes")
plt.tight_layout(); plt.show()
